# 🔱 VoiceBatch Studio v2.9.0 - [Python 3.12 Fixed]
इस वर्जन में Google AI के बताए गए 'Coqpit' और 'ImportError' को फिक्स कर दिया गया है।

In [ ]:
# @title 🛠️ Step 1: पक्का इंस्टॉलेशन (Coqpit Fix)
import os
from IPython.display import display, Javascript

# Anti-Sleep Script
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ पुराने कॉन्फ्लिक्ट्स हटाए जा रहे हैं...")
!pip uninstall -q coqpit coqui-tts -y

print("⏳ नई लाइब्रेरीज़ इंस्टॉल हो रही हैं (Coqpit-Config के साथ)...")
# Google AI के सुझाव के अनुसार सही लाइब्रेरीज़ का चुनाव
!pip install -q coqpit-config
!pip install -q coqui-tts
!pip install -q gradio librosa soundfile

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("✅ सिस्टम अपडेट हो गया! अब Step 2 चलाएं।")

In [ ]:
# @title 🚀 Step 2: Unlimited Voice Studio लॉन्च करें
import gradio as gr
import torch, librosa, os, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"

print("⏳ मॉडल को ड्राइव से लोड किया जा रहा है...")
try:
    tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)
    print("✅ इंजन सफलतापूर्वक चालू हो गया!")
except Exception as e:
    print(f"❌ लोड करने में एरर: {e}")

def voice_engine(text, audio_sample):
    # No Character Limit Logic
    parts = re.split(r'(?<=[।?!])\s+', text)
    combined = []
    for p in parts:
        if len(p.strip()) < 2: continue
        tts.tts_to_file(text=p, speaker_wav=audio_sample, language='hi', file_path='temp.wav')
        y, _ = librosa.load('temp.wav', sr=24000)
        combined.extend(y)
    
    os.makedirs("outputs", exist_ok=True)
    sf.write('outputs/final.wav', np.array(combined), 24000)
    return 'outputs/final.wav'

demo = gr.Interface(fn=voice_engine, 
                    inputs=[gr.Textbox(label="Script (Unlimited)", lines=10), 
                            gr.Audio(label="Upload Sample", type='filepath')], 
                    outputs=gr.Audio(label="Download Result"))
demo.launch(share=True)